# DDoS Traffic Classification — TensorFlow LSTM + Attention

Multiclass klasifikacija mreznog saobracaja pomocu dvostrukog LSTM-a sa Attention mehanizmom.  
Koristimo TimeSeriesSplit kros-validaciju i izvozimo metrike u JSON.

## 1. Importi i konfiguracija

In [ ]:
%%writefile functions.py
import random
import math
import csv
import torch
import sys
import numpy as np

from collections import defaultdict
from datetime import datetime, timezone
from zoneinfo import ZoneInfo

# Featurei koji su u opsegu 0-1
RATIO_FEATURES = {
    "udp_ratio", "tcp_ratio", "icmp_ratio",
    "tcp_syn_ratio", "tcp_ack_ratio", "tcp_fin_ratio",
    "dns_query_ratio", "dns_response_ratio",
    "top_src_ip_packet_share", "top_src_ip_byte_share",
    "top_dst_port_share", "dst_subnet_spread",
}

# Featurei koji moraju biti pozitivni
POSITIVE_FEATURES = {
    "packet_rate", "byte_rate", "avg_packet_size", "std_packet_size",
    "unique_src_ips", "unique_dst_ips", "unique_flows",
    "src_ip_entropy", "dst_ip_entropy", "dst_port_entropy",
}

def rand_uniform(min_val: float, max_val: float) -> float:
    return min_val + random.random() * (max_val - min_val)

# Box-Muller transformation
def rand_normal(mean: float, std: float) -> float:
    u1 = random.random()
    u2 = random.random()
    z = math.sqrt(-2 * math.log(u1)) * math.cos(2 * math.pi * u2)
    return mean + std * z

def clamp(x: float, low: float, high: float) -> float:
    return max(low, min(high, x))

def write_csv(dataset: list[dict], filename: str) -> None:
    if not dataset:
        print("Dataset is empty, nothing to write.")
        return

    if not filename.endswith(".csv"):
        filename += ".csv"

    fieldnames = list(dataset[0].keys())

    with open(filename, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(dataset)

    print(f"Written {len(dataset)} rows to '{filename}'.")

def format_timestamp(ms: int) -> str:
    try:
        # Konvertuje milisekunde u formatirani string
        dt = datetime.fromtimestamp(ms / 1000)
        dt_final = dt.replace(tzinfo=ZoneInfo('UTC'))
        return dt_final.strftime("%Y-%m-%dT%H:%M:%S")
    except Exception as e:
        print(f'Exception functions | format_timestamp: {e} Line: {sys.exc_info()[2].tb_lineno}')


def add_window_metadata(sample: dict, window_id: int, timestamp: int, active_atk: int) -> dict:
    # F-je koje dodaju vremenske serije podacima
    return {
        **sample,
        "window_id": window_id,
        "timestamp": timestamp,
        "ts_formated": format_timestamp(timestamp),
        "attack_active": int(active_atk),
    }


def compute_class_weights(dataset: list[dict], labels: list[str]) -> dict:
    """
    Racuna inverse-frequency weight po klasi.
    Vraca recnik {label: weight} koji se prosledjuje CrossEntropyLoss-u.
    """
    try:
        counts = {label: 0 for label in labels}
        for sample in dataset:
            lbl = sample.get("label")
            if lbl in counts:
                counts[lbl] += 1

        total = sum(counts.values())
        n_classes = len(labels)

        weights = {}
        for label, count in counts.items():
            weights[label] = total / (n_classes * count) if count > 0 else 1.0

        print("\nClass weights:")
        for label, w in weights.items():
            print(f"  {label:<25} count: {counts[label]:>7}   weight: {w:.4f}")

        return weights

    except Exception as e:
        print(f'Exception functions | compute_class_weights: {e} Line: {sys.exc_info()[2].tb_lineno}')


def get_class_weights_tensor(dataset: list[dict], labels: list[str], device) -> "torch.Tensor":
    try:
        weights_dict = compute_class_weights(dataset, labels)
        weights_list = [weights_dict.get(label, 1.0) for label in labels]
        return torch.tensor(weights_list, dtype=torch.float32).to(device)
    except Exception as e:
        print(f'Exception functions | get_class_weights_tensor: {e} Line: {sys.exc_info()[2].tb_lineno}')


def oversample_minority_classes(
    dataset: list[dict],
    labels: list[str],
    target_ratio: float = 0.5,
) -> list[dict]:
    """
    Oversampluje manjinske klase do target_ratio * count(normal).
    target_ratio=0.5 znaci da svaka napadna klasa ima bar 50% uzoraka normal klase.
    Koristi add_noise za male varijacije pri dupliranju uzoraka.
    """
    try:
        by_label = defaultdict(list)
        for sample in dataset:
            by_label[sample["label"]].append(sample)

        normal_count = len(by_label.get("normal", []))
        target_count = int(normal_count * target_ratio)

        print(f"\nOversampling — target by class: {target_count} (normal: {normal_count})")

        oversampled = list(dataset)

        for label in labels:
            if label == "normal":
                continue

            current = by_label.get(label, [])
            current_count = len(current)

            if current_count >= target_count:
                print(f"  {label:<25} {current_count:>7} skipping this")
                continue

            needed = target_count - current_count
            print(f"  {label:<25} {current_count:>7} > adding {needed} sample")

            for i in range(needed):
                base = random.choice(current)
                noisy = add_noise(base, noise_level=0.04)
                noisy["label"]         = label
                noisy["attack_active"] = base["attack_active"]
                noisy["window_id"]     = base["window_id"]
                noisy["timestamp"]     = base["timestamp"] + i
                noisy["ts_formated"]   = format_timestamp(noisy["timestamp"])
                oversampled.append(noisy)

        return oversampled

    except Exception as e:
        print(f'Exception functions | oversample_minority_classes: {e} Line: {sys.exc_info()[2].tb_lineno}')


def add_noise(sample: dict, noise_level: float = 0.05) -> dict:
    """
    Dodaje Gaussov sum na numericke featuere.
    noise_level = standardna devijacija kao procenat vrednosti featura.
    """
    try:
        noisy = dict(sample)
        for key, val in sample.items():
            if not isinstance(val, (int, float)):
                continue
            if key in ("window_id", "timestamp", "attack_active"):
                continue

            noise = np.random.normal(0, abs(val) * noise_level + 1e-6)
            noisy_val = val + noise

            if key in RATIO_FEATURES:
                noisy_val = float(np.clip(noisy_val, 0.0, 1.0))
            elif key in POSITIVE_FEATURES:
                noisy_val = max(0.0, noisy_val)

            noisy[key] = noisy_val

        return noisy

    except Exception as e:
        print(f'Exception functions | add_noise: {e} Line: {sys.exc_info()[2].tb_lineno}')


def blend_samples(sample_a: dict, sample_b: dict, alpha: float) -> dict:
    """
    Linearno interpoluje izmedju dva uzorka.
    alpha=0.0 -> sample_a, alpha=1.0 -> sample_b
    Koristi se za tranzicione periode.
    """
    try:
        blended = dict(sample_a)
        for key, val_a in sample_a.items():
            if not isinstance(val_a, (int, float)):
                continue
            if key in ("window_id", "timestamp", "attack_active", "label"):
                continue
            val_b = sample_b.get(key, val_a)
            blended[key] = val_a * (1 - alpha) + val_b * alpha
        return blended
    except Exception as e:
        print(f'Exception functions | blend_samples: {e} Line: {sys.exc_info()[2].tb_lineno}')


def generate_transition(from_fn, to_fn, from_label: str, to_label: str, n_windows: int, start_ts: int, window_ms: int) -> list[dict]:
    """
    Generise tranzicioni period izmedju dva tipa saobracaja.
    Prvih 50% prozora: label from_label sa rastucim alpha ka to_fn
    Drugih 50% prozora: label to_label sa opadajucim alpha
    """
    try:
        result = []
        for i in range(n_windows):
            alpha  = i / n_windows
            base_a = from_fn()
            base_b = to_fn()

            blended = blend_samples(base_a, base_b, alpha)
            blended = add_noise(blended, noise_level=0.08)

            label = from_label if alpha < 0.5 else to_label
            ts    = start_ts + i * window_ms

            result.append({
                **blended,
                "label":          label,
                "window_id":      i,
                "timestamp":      ts,
                "ts_formated":    format_timestamp(ts),
                "attack_active":  1 if label != "normal" else 0,
            })

        return result

    except Exception as e:
        print(f'Exception functions | generate_transition: {e} Line: {sys.exc_info()[2].tb_lineno}')

In [ ]:
%%writefile metrics_exporter.py
import json
import logging
import sys
from datetime import datetime
from pathlib import Path
from typing import Optional

import numpy as np
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    matthews_corrcoef,
    roc_auc_score,
)

logger = logging.getLogger(__name__)


def export_metrics_to_json(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    y_proba: np.ndarray,
    class_labels: list[str],
    output_dir: str = ".",
    filename: str = "eval_metrics.json",
) -> str:
    """
    Racuna sve metrike evaluacije i cuva ih u JSON fajl.

    Parametri:
        y_true       — stvarne oznake (integer indeksi klasa)
        y_pred       — predvidjene oznake (integer indeksi klasa)
        y_proba      — verovatnoca po klasi (shape: n_samples x n_classes)
        class_labels — lista naziva klasa u ispravnom redosledu
        output_dir   — direktorijum za cuvanje JSON fajla
        filename     — naziv izlaznog fajla

    Vraca:
        Apsolutnu putanju do sacuvanog JSON fajla.
    """
    try:
        report = classification_report(
            y_true, y_pred,
            target_names=class_labels,
            output_dict=True,
            zero_division=0,
        )
        # Cuvamo samo per-class stavke (iskljucujemo accuracy, macro avg, weighted avg)
        per_class_report = {
            cls: metrics
            for cls, metrics in report.items()
            if isinstance(metrics, dict)
        }

        cm  = confusion_matrix(y_true, y_pred)
        mcc = float(matthews_corrcoef(y_true, y_pred))
        roc_auc = _compute_roc_auc_per_class(y_true, y_proba, class_labels)

        metrics_payload = {
            "timestamp":             datetime.now().isoformat(),
            "class_labels":          class_labels,
            "mcc_score":             mcc,
            "classification_report": per_class_report,
            "confusion_matrix":      cm.tolist(),
            "roc_auc_scores":        roc_auc,
            "summary": {
                "macro_f1":        report.get("macro avg",    {}).get("f1-score",  0.0),
                "weighted_f1":     report.get("weighted avg", {}).get("f1-score",  0.0),
                "macro_precision": report.get("macro avg",    {}).get("precision", 0.0),
                "macro_recall":    report.get("macro avg",    {}).get("recall",    0.0),
                "total_samples":   int(report.get("macro avg", {}).get("support",  0)),
            },
        }

        output_path = Path(output_dir) / filename
        output_path.parent.mkdir(parents=True, exist_ok=True)

        with open(output_path, "w", encoding="utf-8") as f:
            json.dump(metrics_payload, f, indent=2)

        logger.info(f"Metrics saved to: {output_path}")
        return str(output_path.resolve())

    except Exception as e:
        logger.error(f"Failed to export metrics: {e} | Line: {sys.exc_info()[2].tb_lineno}")
        raise


def _compute_roc_auc_per_class(y_true: np.ndarray, y_proba: np.ndarray, class_labels: list[str]) -> dict:
    """
    Racuna ROC-AUC za svaku klasu metodom one-vs-rest.
    Vraca 0.0 za klasu koja nije prisutna u y_true (ne moze se izracunati).
    """
    roc_auc: dict = {}

    for i, label in enumerate(class_labels):
        if i >= y_proba.shape[1]:
            roc_auc[label] = 0.0
            continue

        y_true_binary = (y_true == i).astype(int)

        if y_true_binary.sum() == 0:
            logger.warning(f"Class '{label}' not present in y_true — ROC-AUC set to 0.0")
            roc_auc[label] = 0.0
            continue

        try:
            roc_auc[label] = float(roc_auc_score(y_true_binary, y_proba[:, i]))
        except Exception as e:
            logger.warning(f"ROC-AUC failed for class '{label}': {e}")
            roc_auc[label] = 0.0

    return roc_auc


def load_metrics_from_json(path: str) -> Optional[dict]:
    """
    Ucitava prethodno sacuvane metrike iz JSON fajla.
    Vraca None ako fajl ne postoji ili je ostecen.
    """
    p = Path(path)
    if not p.exists():
        logger.warning(f"Metrics file not found: {path}")
        return None

    try:
        with open(p, "r", encoding="utf-8") as f:
            return json.load(f)
    except (json.JSONDecodeError, OSError) as e:
        logger.error(f"Failed to load metrics from '{path}': {e}")
        return None

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, optimizers, losses
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_auc_score,
    roc_curve,
    matthews_corrcoef
)
from sklearn.preprocessing import label_binarize
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os
from functions import *
from metrics_exporter import export_metrics_to_json

In [ ]:
# Hiperparametri
SEQUENCE_LEN  = 50
BATCH_SIZE    = 64
HIDDEN_SIZE   = 128
NUM_LAYERS    = 2
DROPOUT       = 0.3
LEARNING_RATE = 1e-3
EPOCHS        = 20
N_SPLITS      = 5   # Broj foldova za TimeSeriesSplit

EXCLUDED_COLS = ['label', 'window_id', 'timestamp', 'ts_formated', 'attack_active', 'instance_id', 'vector_id']

LABELS = [
    'normal',
    'udp_flood_large',
    'dns_amplification',
    'subnet_carpet_bombing',
    'syn_flood',
    'icmp_flood',
    'udp_flood_mixed',
    'ntp_amplification',
    'ack_flood'
]

## 2. GPU konfiguracija

In [ ]:
# Moja verzija tf nije podrzavala treniranje na GPU
# Moralo je preko wlsa da se podesi
# Pokusao sam preko dockera da resim ovaj problem ali kad pokrenem ovu skriptu iz docker okruzenja i dalje nije prepoznavao GPU
# Nisam siguran zasto, jer ollama model bez problema koristi gpu.
gpus = tf.config.list_physical_devices('GPU')

if gpus:
    print(f"Found GPU: {gpus}")
    try:
        # Ukljucivanje 'Memory Growth' opcije
        # Ovo sprecava TF da odmah zauzme 100% VRAM-a, vec alocira memoriju po potrebi.
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("Memory growth za GPU is enabled.")
    except RuntimeError as e:
        # Memory growth mora biti postavljen pre inicijalizacije GPU-a
        print(e)
else:
    print("GPU not found, training on CPU.")

## 3. Priprema podataka

In [ ]:
def prepare_data(csv_path: str, seq_len: int = SEQUENCE_LEN):
    """
    Ucitava CSV, sortira po timestamp-u, skalira feature i pravi sekvence.
    """
    try:
        df = pd.read_csv(csv_path)

        if "timestamp" in df.columns:
            df = df.sort_values("timestamp").reset_index(drop=True)

        feature_cols = [c for c in df.columns if c not in EXCLUDED_COLS]
        X_raw = df[feature_cols].values.astype(np.float32)

        label_enc = LabelEncoder()
        label_enc.classes_ = np.array(LABELS)
        Y_raw = label_enc.transform(df["label"].values)

        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X_raw)

        sequences, labels = [], []
        for i in range(len(X_scaled) - seq_len):
            sequences.append(X_scaled[i: i + seq_len])
            labels.append(Y_raw[i + seq_len - 1])

        return (
            np.array(sequences),
            np.array(labels),
            scaler,
            label_enc,
            feature_cols,
        )
    except Exception as e:
        print(f"Exception | prepare_data: {e} Line: {sys.exc_info()[2].tb_lineno}")

## 4. Definicija modela

### 4.1 Attention Layer

In [ ]:
class AttentionLayer(layers.Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def build(self, input_shape):
        # Nauceni vektor paznje
        self.attention = layers.Dense(1, use_bias=False)
        super().build(input_shape)

    def call(self, lstm_out):
        """
        lstm_out: (batch, seq_len, hidden_size)
        Vraca:
            context: (batch, hidden_size)
            weights: (batch, seq_len)
        """
        # Skorovi paznje za svaki timestep
        scores  = self.attention(lstm_out)               # (batch, seq_len, 1)
        weights = tf.nn.softmax(scores, axis=1)          # (batch, seq_len, 1)

        # Tezinski zbir hidden stateova
        context = tf.reduce_sum(weights * lstm_out, axis=1)  # (batch, hidden_size)
        weights = tf.squeeze(weights, axis=-1)               # (batch, seq_len)

        return context, weights

### 4.2 DDoS LSTM + Attention Model

In [ ]:
class DDoSLSTMAttention(models.Model):
    def __init__(self, hidden_size: int, num_layers: int, num_classes: int, dropout: float, **kwargs):
        super().__init__(**kwargs)

        self.lstm_layers = []
        for i in range(num_layers):
            # U Keras-u moramo vratiti sekvence da bi Attention radio pravilno,
            # i za prosledjivanje narednom LSTM sloju
            self.lstm_layers.append(layers.LSTM(hidden_size, return_sequences=True))
            if i < num_layers - 1 and dropout > 0.0:
                self.lstm_layers.append(layers.Dropout(dropout))

        self.attention = AttentionLayer()

        self.classifier = models.Sequential([
            layers.Dropout(dropout),
            layers.Dense(hidden_size // 2, activation='relu'),
            layers.Dense(num_classes)  # Logits (bez softmax aktivacije, softmax koristimo pri evaluaciji)
        ])

    def call(self, inputs, training=False):
        x = inputs
        for layer in self.lstm_layers:
            x = layer(x, training=training)

        context, weights = self.attention(x)
        logits = self.classifier(context, training=training)
        return logits, weights

    # Override metoda da bi keras .fit() ignorisao 'weights' tokom racunanja loss-a
    def train_step(self, data):
        x, y = data
        with tf.GradientTape() as tape:
            logits, _ = self(x, training=True)
            loss = self.compiled_loss(y, logits, regularization_losses=self.losses)

        trainable_vars = self.trainable_variables
        gradients = tape.gradient(loss, trainable_vars)
        self.optimizer.apply_gradients(zip(gradients, trainable_vars))
        self.compiled_metrics.update_state(y, logits)
        return {m.name: m.result() for m in self.metrics}

    def test_step(self, data):
        x, y = data
        logits, _ = self(x, training=False)
        loss = self.compiled_loss(y, logits, regularization_losses=self.losses)
        self.compiled_metrics.update_state(y, logits)
        return {m.name: m.result() for m in self.metrics}

## 5. Pomocne funkcije za vizualizaciju

In [ ]:
def plot_confusion_matrix(y_true, y_pred, label_names: list[str]):
    try:
        os.makedirs("graphs", exist_ok=True)
        cm = confusion_matrix(y_true, y_pred)
        cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

        fig, axes = plt.subplots(1, 2, figsize=(20, 8))
        for ax, data, title, fmt in zip(
            axes, [cm, cm_norm],
            ["Confusion Matrix (absolute numbers)", "Confusion Matrix (normalized)"],
            ["d", ".2f"],
        ):
            sns.heatmap(
                data, annot=True, fmt=fmt, cmap="Blues",
                xticklabels=label_names, yticklabels=label_names,
                ax=ax, linewidths=0.5,
            )
            ax.set_title(title, fontsize=13, pad=12)
            ax.set_xlabel("Predicted", fontsize=11)
            ax.set_ylabel("Actual",    fontsize=11)
            ax.tick_params(axis="x", rotation=45)
            ax.tick_params(axis="y", rotation=0)

        plt.tight_layout()
        plt.savefig("graphs/confusion_matrix_tf.png", dpi=150)
        plt.show()
        print("Saved confusion matrix -> graphs/confusion_matrix_tf.png")
    except Exception as e:
        print(f'Exception | plot_confusion_matrix: {e}')

In [ ]:
def plot_roc_curves(y_bin, all_probs, label_names: list[str]):
    try:
        os.makedirs("graphs", exist_ok=True)
        n_classes = len(label_names)
        colors = plt.cm.tab10(np.linspace(0, 1, n_classes))

        plt.figure(figsize=(10, 7))
        for i, (name, color) in enumerate(zip(label_names, colors)):
            fpr, tpr, _ = roc_curve(y_bin[:, i], all_probs[:, i])
            auc = roc_auc_score(y_bin[:, i], all_probs[:, i])
            plt.plot(fpr, tpr, color=color, lw=1.8, label=f"{name}  (AUC = {auc:.3f})")

        plt.plot([0, 1], [0, 1], "k--", lw=1)
        plt.xlabel("False Positive Rate", fontsize=12)
        plt.ylabel("True Positive Rate",  fontsize=12)
        plt.title("ROC curves by class (one-vs-rest)", fontsize=13)
        plt.legend(loc="lower right", fontsize=9)
        plt.tight_layout()
        plt.savefig("graphs/roc_curves_tf.png", dpi=150)
        plt.show()
        print("Saved ROC curve -> graphs/roc_curves_tf.png")
    except Exception as e:
        print(f'Exception | plot_roc_curves: {e}')

## 6. Kros-validacija i trening

In [ ]:
def cross_validate(csv_path: str, save_path: str = "ddos_lstm_attention.keras"):
    try:
        sequences, labels, scaler, le, feature_cols = prepare_data(csv_path)
        num_features = len(feature_cols)
        num_classes  = len(le.classes_)

        tss = TimeSeriesSplit(n_splits=N_SPLITS)
        fold_metrics = []
        best_overall_loss = float("inf")

        for fold, (train_idx, val_idx) in enumerate(tss.split(sequences), start=1):
            print(f"\n{'='*60}")
            print(f"  Fold {fold}/{N_SPLITS} | Train: {len(train_idx):,}  Val: {len(val_idx):,}")
            print(f"{'='*60}")

            X_train, y_train = sequences[train_idx], labels[train_idx]
            X_val,   y_val   = sequences[val_idx],   labels[val_idx]

            # Koriscenje tf.data API-ja za optimizovan data pipeline
            train_ds = (
                tf.data.Dataset
                .from_tensor_slices((X_train, y_train))
                .batch(BATCH_SIZE)
                .prefetch(tf.data.AUTOTUNE)
            )
            val_ds = (
                tf.data.Dataset
                .from_tensor_slices((X_val, y_val))
                .batch(BATCH_SIZE)
                .prefetch(tf.data.AUTOTUNE)
            )

            # Inicijalizacija modela (nova instanca za svaki fold)
            model = DDoSLSTMAttention(
                hidden_size=HIDDEN_SIZE,
                num_layers=NUM_LAYERS,
                num_classes=num_classes,
                dropout=DROPOUT
            )

            # Gradient clipping je integrisan u optimizer
            optimizer = optimizers.Adam(learning_rate=LEARNING_RATE, clipnorm=1.0)

            # from_logits=True posto nas poslednji Dense sloj nema aktivaciju
            model.compile(
                optimizer=optimizer,
                loss=losses.SparseCategoricalCrossentropy(from_logits=True),
                metrics=['accuracy']
            )

            lr_scheduler = callbacks.ReduceLROnPlateau(
                monitor='val_loss', patience=3, factor=0.5, verbose=1
            )

            # Treniranje folda
            history = model.fit(
                train_ds,
                validation_data=val_ds,
                epochs=EPOCHS,
                callbacks=[lr_scheduler],
                verbose=1
            )

            # Manualna evaluacija na validacionom setu za racunanje napredne metrike
            val_logits, val_weights = model.predict(val_ds)
            val_probs = tf.nn.softmax(val_logits, axis=-1).numpy()
            val_preds = np.argmax(val_probs, axis=-1)

            v_loss = history.history['val_loss'][-1]
            v_acc  = history.history['val_accuracy'][-1]
            mcc    = matthews_corrcoef(y_val, val_preds)

            metrics = {
                "fold":    fold,
                "val_loss": v_loss,
                "val_acc":  v_acc,
                "mcc":      mcc,
                "preds":    val_preds,
                "targets":  y_val,
                "probs":    val_probs,
            }
            fold_metrics.append(metrics)

            if v_loss < best_overall_loss:
                best_overall_loss = v_loss
                # Cuvamo model u nativnom Keras formatu
                model.save_weights(save_path)
                print(f"  New best model saved (fold {fold}) -> {save_path}")

        # Sumarni rezultati
        print(f"\n{'='*60}")
        print("  Cross-validation summary")
        print(f"{'='*60}")
        print(f"  {'Fold':<8} {'Val Loss':<12} {'Val Acc':<12} {'MCC'}")
        print(f"  {'-'*48}")

        for m in fold_metrics:
            print(f"  {m['fold']:<8} {m['val_loss']:<12.4f} {m['val_acc']:<12.4f} {m['mcc']:.4f}")

        avg_loss = np.mean([m["val_loss"] for m in fold_metrics])
        avg_acc  = np.mean([m["val_acc"]  for m in fold_metrics])
        avg_mcc  = np.mean([m["mcc"]      for m in fold_metrics])
        std_acc  = np.std( [m["val_acc"]  for m in fold_metrics])

        print(f"  {'-'*48}")
        print(f"  {'Avg':<8} {avg_loss:<12.4f} {avg_acc:<12.4f} {avg_mcc:.4f}")
        print(f"  {'Std':<8} {'':12} {std_acc:<12.4f}")
        print(f"{'='*60}\n")

        # Evaluacija najboljeg folda
        best_fold    = min(fold_metrics, key=lambda m: m["val_loss"])
        best_targets = np.array(best_fold["targets"])
        best_preds   = np.array(best_fold["preds"])
        best_probs   = best_fold["probs"]

        print(f"Classification report (best fold {best_fold['fold']}):")
        print(classification_report(
            best_targets, best_preds,
            labels=list(range(num_classes)),
            target_names=LABELS,
            digits=4,
            zero_division=0,
        ))
        print(f"Matthews Correlation Coefficient (best fold {best_fold['fold']}): {best_fold['mcc']:.4f}\n")

        # Vizualizacija
        plot_confusion_matrix(best_targets, best_preds, LABELS)
        y_bin = label_binarize(best_targets, classes=list(range(num_classes)))
        plot_roc_curves(y_bin, best_probs, LABELS)

        # Export metrika u JSON
        print("Exporting evaluation metrics to JSON...")
        os.makedirs("results", exist_ok=True)
        json_path = export_metrics_to_json(
            y_true=best_targets,
            y_pred=best_preds,
            y_proba=best_probs,
            class_labels=LABELS,
            output_dir="results/",
            filename="eval_metrics_tf.json",
        )
        print(f"Metrics saved -> {json_path}")
        print("Training finished!")

        return fold_metrics

    except Exception as e:
        print(f"Exception | cross_validate: {e} Line: {sys.exc_info()[2].tb_lineno}")

## 7. Pokretanje treninga

In [ ]:
# Unesi putanju do CSV fajla
csv_path = input('Insert csv file path: ').strip()
fold_metrics = cross_validate(csv_path)